## Dependencies

In [ ]:
## libraries
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.evaluators.config import FEAT_X, FEAT_Z, TARGET
from src.evaluators.config import load_data, load_models
from src.evaluators.falsifying import eval_falsifiability

## Initialization

In [ ]:
## setup
data = load_data()
models = load_models()

## Falsifiability Tests

In [ ]:
## run falsifiability evaluation: real vs falsified data under logo-cv
falsifiability = eval_falsifiability(
    data = data,
    models = models,
    feat_x = FEAT_X,
    feat_z = FEAT_Z,
    target = TARGET,
    random_state = 42,
    n_jobs = -1
)

In [ ]:
from scipy.stats import wilcoxon

## display settings
pd.set_option("display.float_format", lambda x: f"{x:.6f}")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)

## --- aggregate to domain level: mean EI per (model, method, condition, group) ---
domain_ei = falsifiability.groupby(
    ["model", "method", "condition", "group"]
)["ei"].mean().reset_index()

## --- pairwise wilcoxon: real vs falsified per method (n = 45) ---
methods = falsifiability["method"].unique()
rows = []

for method in methods:
    real_ei = domain_ei.query(
        "method == @method and condition == 'real'"
    ).set_index(["model", "group"])["ei"]

    false_ei = domain_ei.query(
        "method == @method and condition == 'falsified'"
    ).set_index(["model", "group"])["ei"]

    delta = (real_ei - false_ei).dropna()
    n = len(delta)
    n_real_better = int((delta > 0).sum())
    n_false_better = int((delta <= 0).sum())

    stat, p = wilcoxon(delta.values, alternative = "greater")
    r_effect = 1 - (2 * stat) / (n * (n + 1) / 2)

    if p < 0.01:
        sig = "**"
    elif p < 0.05:
        sig = "*"
    elif p < 0.10:
        sig = "†"
    else:
        sig = ""

    rows.append({
        "Method": method,
        "n": n,
        "Real Better": n_real_better,
        "Falsified Better": n_false_better,
        "Mean Δ EI": delta.mean(),
        "Median Δ EI": delta.median(),
        "Effect Size (r)": r_effect,
        "p (one-sided)": p,
        "Sig.": sig,
    })

result = pd.DataFrame(rows)

print(f"=== Falsifiability: Real vs Falsified Data (n = {n}) ===")
print("H₁: Real data produces higher frontier EI than falsified data")
print("Each row tests one falsification method against real data\n")
display(
    result.style
        .format({
            "Mean Δ EI": "{:+.4f}",
            "Median Δ EI": "{:+.4f}",
            "Effect Size (r)": "{:+.3f}",
            "p (one-sided)": "{:.4f}",
        })
        .set_caption("** p < 0.01, * p < 0.05, † p < 0.10")
        .hide(axis = "index")
)

In [ ]:
## --- mean frontier metrics: real vs falsified by method ---
print("=== Mean Frontier Metrics by Method and Condition ===\n")
summary = falsifiability.groupby(
    ["method", "condition"]
)[["ei", "vr", "mv", "ms", "ea"]].mean()
display(summary.round(4))

## --- overall falsification summary ---
n_methods = len(result)
n_sig = int((result["p (one-sided)"] < 0.05).sum())
mean_effect = result["Effect Size (r)"].mean()
worst_p = result["p (one-sided)"].max()

print(f"\nReal data outperforms {n_sig}/{n_methods} falsification methods (p < 0.05)")
print(f"Mean effect size: r = {mean_effect:+.3f}")
print(f"Worst p-value: {worst_p:.4f}")